# Four-region benchmark

Two assignment runs (Fashion-MNIST + pixel MLP, CIFAR-10 + ResNet-18) with the default
four-region class partition. YAML mirrors: `configs/experiment/four_region_fashion_mlp.yaml`,
`configs/experiment/four_region_cifar_resnet.yaml`.

**Attribution backends** (cell 2 toggles):
- **DualXDA** — always on
- **GradDot** — `ENABLE_GRADDOT` (off by default; slow on ResNet, ~30–60 min)
- **EK-FAC** — `ENABLE_EK_FAC` (needs `pip install kronfluence`)

Artifacts: `per_sample_signals.csv`, `zwischen/02_influence_*.pt` (full `[eval × train]` matrices).

If a run crashed mid-way, delete `<run_dir>/cache/graddot/` or `cache/ek_fak/` and re-run cell 2.

In [3]:
from copy import deepcopy
from pathlib import Path
import importlib.util
import sys

%load_ext autoreload
%autoreload 2


def _bootstrap_uqlab() -> Path:
    for base in (Path.cwd(), *Path.cwd().parents):
        for path in (
            base / "notebooks" / "bootstrap_uqlab.py",
            base / "uqlab-streamlit" / "notebooks" / "bootstrap_uqlab.py",
            base / "four-region-benchmark" / "notebooks" / "bootstrap_uqlab.py",
        ):
            if not path.is_file():
                continue
            spec = importlib.util.spec_from_file_location("_uqlab_bootstrap", path)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod.ensure_uqlab_path()
    raise ModuleNotFoundError(
        "bootstrap_uqlab.py not found — open uqlab-streamlit/notebooks/ in Jupyter"
    )


PROJECT_ROOT = _bootstrap_uqlab()
for _mod in list(sys.modules):
    if _mod == "uqlab" or _mod.startswith("uqlab."):
        del sys.modules[_mod]

from uqlab.data.class_regions import DEFAULT_FOUR_REGION_PRESET
from uqlab.data.packs import prepare_run_data_context
from uqlab.data.setup import prepare_experiment_data
from uqlab.evaluation.reporting.four_region_reporting import (
    list_four_region_signal_columns,
    plot_four_region_metrics_by_group,
)
from uqlab.runtime_paths import repository_root
from uqlab.runner.phases.config_view import apply_data_context, extract_run_config
from uqlab.runner.train_eval import run_train_and_eval_phases
from uqlab.shared.config.classification import ExperimentConfig
from uqlab.shared.utils.classification import auto_device, set_seed

ROOT = repository_root()
RESULTS_BASE = ROOT / "results/four_region_benchmark"
SEED = 42

set_seed(SEED)
device = auto_device("auto")
RESULTS_BASE.mkdir(parents=True, exist_ok=True)

import importlib.util

HAS_KRONFLUENCE = importlib.util.find_spec("kronfluence") is not None
print("Root:", ROOT)
print("Device:", device)
print("Kronfluence (EK-FAC):", "installed" if HAS_KRONFLUENCE else "not installed — pip install kronfluence")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Root: /Users/andrearachetta/Documents/old_pilots/uqlab-streamlit
Device: mps
Kronfluence (EK-FAC): installed


In [5]:
# --- edit here: runs, regions, attribution toggles ---
ENABLE_GRADDOT = False  # GradDot: slow on CIFAR ResNet (~30–60 min); uses 128-dim projection
ENABLE_EK_FAC = HAS_KRONFLUENCE  # EK-FAC; set True after: pip install kronfluence

_DUALXDA_METRICS = [
    "inverse_coherence_dualxda",
    "inverse_mass_dualxda",
    "inverse_dominance_dualxda",
]
_GRADDOT_METRICS = [
    "inverse_coherence_graddot",
    "inverse_mass_graddot",
    "inverse_dominance_graddot",
]
_EK_FAC_METRICS = [
    "inverse_coherence_ek_fak",
    "inverse_mass_ek_fak",
    "inverse_dominance_ek_fak",
]
_PREDICTIVE_METRICS = [
    "expected_entropy",
    "mutual_info",
    "inverse_logit_magnitude",
]


def apply_attribution_config(cfg: ExperimentConfig) -> None:
    backends = ["dualxda"]
    attribution = list(_DUALXDA_METRICS)
    if ENABLE_GRADDOT:
        backends.append("graddot")
        attribution.extend(_GRADDOT_METRICS)
    if ENABLE_EK_FAC:
        if not HAS_KRONFLUENCE:
            raise ImportError(
                "ENABLE_EK_FAC=True but kronfluence is not installed. "
                "Run: pip install kronfluence"
            )
        backends.append("ek_fak")
        attribution.extend(_EK_FAC_METRICS)
    cfg.evaluation.attribution_backends = backends
    cfg.evaluation.signals["attribution"] = attribution


def build_plot_metrics() -> list[str]:
    """Primary metrics for box plots (one per backend + baselines)."""
    metrics = ["inverse_coherence_dualxda"]
    if ENABLE_EK_FAC:
        metrics.append("inverse_coherence_ek_fak")
    if ENABLE_GRADDOT:
        metrics.append("inverse_coherence_graddot")
    metrics.extend(_PREDICTIVE_METRICS)
    return metrics


RUNS = [
    {
        "name": "fashion_mlp",
        "config_path": ROOT / "configs/experiment/four_region_fashion_mlp.yaml",
        "class_regions": DEFAULT_FOUR_REGION_PRESET,
    },
    {
        "name": "cifar_resnet",
        "config_path": ROOT / "configs/experiment/four_region_cifar_resnet.yaml",
        "class_regions": DEFAULT_FOUR_REGION_PRESET,
    },
]

ATTRIBUTION_METRICS = build_plot_metrics()
print(
    "Attribution:",
    "dualxda",
    "+ graddot" if ENABLE_GRADDOT else "(graddot off)",
    "+ ek_fak" if ENABLE_EK_FAC else "(ek_fak off)",
)


def run_four_region_entry(entry: dict) -> Path:
    cfg = ExperimentConfig.from_yaml(entry["config_path"])
    cfg.data.partition_mode = "four_region"
    cfg.data.class_regions = deepcopy(entry["class_regions"])
    apply_attribution_config(cfg)

    run_dir = RESULTS_BASE / entry["name"]
    run_dir.mkdir(parents=True, exist_ok=True)
    run_cache_dir = run_dir / "cache"

    run_cfg = extract_run_config(cfg)
    data_ctx = prepare_experiment_data(cfg, ROOT, seed=SEED)
    apply_data_context(run_cfg, data_ctx)
    data_pack = prepare_run_data_context(
        config=cfg,
        dataset=data_ctx.dataset,
        split_spec=data_ctx.split_spec,
        dataset_name=run_cfg.dataset_name,
        device=device,
        feature_cache_dir=ROOT / cfg.paths.feature_cache_dir,
        noise_type=run_cfg.noise_type,
        feature_batch_size=run_cfg.feature_batch_size,
    )

    run_train_and_eval_phases(
        config=cfg,
        run_cfg=run_cfg,
        results_dir=run_dir,
        run_cache_dir=run_cache_dir,
        data_pack=data_pack,
        split_spec=data_ctx.split_spec,
        device=device,
        seed=SEED,
        training_config=cfg.training,
        data_config=cfg.data,
        model_config=cfg.model,
        eval_config=cfg.evaluation,
        ds_spec=run_cfg.dataset_spec,
        config_path=entry["config_path"],
        persist=True,
        log=True,
    )
    return run_dir


completed_dirs = []
for spec in RUNS:
    print(f"=== {spec['name']} ===")
    completed_dirs.append(run_four_region_entry(spec))
print("Done:", completed_dirs)

Attribution: dualxda (graddot off) + ek_fak
=== fashion_mlp ===
Epoch 1/12, Loss: 1.5461
Epoch 2/12, Loss: 1.1220
Epoch 3/12, Loss: 1.0491
Epoch 4/12, Loss: 1.0176
Epoch 5/12, Loss: 0.9934
Epoch 6/12, Loss: 0.9851
Epoch 7/12, Loss: 0.9787
Epoch 8/12, Loss: 0.9476
Epoch 9/12, Loss: 0.9218
Epoch 10/12, Loss: 0.9145
Epoch 11/12, Loss: 0.8909
Epoch 12/12, Loss: 0.8744
Deterministic eval forward (DualXDA targets)...
Attribution primitives (ek_fak, dualxda)...
MC Dropout (10 passes, batched eval)...
Using DualXDA classifier layer: classifier
EK-FAC: fitting Kronfluence factors and computing pairwise scores...
EK-FAC: running on cpu (Kronfluence does not support mps)


/Users/andrearachetta/Documents/old_pilots/.venv/lib/python3.14/site-packages/kronfluence/factor/eigen.py:398: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)
/Users/andrearachetta/Documents/old_pilots/.venv/lib/python3.14/site-packages/kronfluence/score/pairwise.py:206: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)


✅ Zwischenergebnisse: /Users/andrearachetta/Documents/old_pilots/uqlab-streamlit/results/four_region_benchmark/fashion_mlp/zwischen/

Noisy eval samples (is_noisy=True): 200 / 800
   dataset_index  group             clean  noisy
           32168  aleatoric_like        2      6
           15351  aleatoric_like        1      6
           56205  aleatoric_like        0      3
           31723  aleatoric_like        0      3
           30542  aleatoric_like        1      8
           55128  aleatoric_like        0      9
            2975  aleatoric_like        0      7
            9891  aleatoric_like        2      4
           22053  aleatoric_like        2      6
           58953  aleatoric_like        2      9
           41078  aleatoric_like        2      5
           55133  aleatoric_like        2      6
           31074  aleatoric_like        3      1
           29183  aleatoric_like        2      1
           15085  aleatoric_like        0      3
           53685  aleatoric_like    

/Users/andrearachetta/Documents/old_pilots/.venv/lib/python3.14/site-packages/kronfluence/factor/covariance.py:200: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)
/Users/andrearachetta/Documents/old_pilots/.venv/lib/python3.14/site-packages/kronfluence/factor/eigen.py:398: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)
/Users/andrearachetta/Documents/old_pilots/.venv/lib/python3.14/site-packages/kronfluence/score/pairwise.py:206: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)


✅ Zwischenergebnisse: /Users/andrearachetta/Documents/old_pilots/uqlab-streamlit/results/four_region_benchmark/cifar_resnet/zwischen/

Noisy eval samples (is_noisy=True): 200 / 800
   dataset_index  group             clean  noisy
           25488  aleatoric_like        1      7
           33097  aleatoric_like        1      5
           28301  aleatoric_like        2      5
           22664  aleatoric_like        3      1
           38179  aleatoric_like        0      3
            9096  aleatoric_like        0      4
           10935  aleatoric_like        2      3
           29001  aleatoric_like        3      8
           24460  aleatoric_like        3      2
           12818  aleatoric_like        0      3
            1831  aleatoric_like        0      4
           29225  aleatoric_like        0      9
           25626  aleatoric_like        2      6
           48318  aleatoric_like        0      9
           46507  aleatoric_like        1      7
            7649  aleatoric_like   

In [ ]:
# --- PLOTS ONLY: run after kernel restart (loads bootstrap from disk) ---
import importlib.util
from pathlib import Path


def _bootstrap_uqlab() -> Path:
    for base in (Path.cwd(), *Path.cwd().parents):
        for path in (
            base / "notebooks" / "bootstrap_uqlab.py",
            base / "uqlab-streamlit" / "notebooks" / "bootstrap_uqlab.py",
            base / "four-region-benchmark" / "notebooks" / "bootstrap_uqlab.py",
        ):
            if not path.is_file():
                continue
            spec = importlib.util.spec_from_file_location("_uqlab_bootstrap", path)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod.ensure_uqlab_path()
    raise ModuleNotFoundError(
        "bootstrap_uqlab.py not found — File → Revert, then re-run this cell"
    )


_bootstrap_uqlab()

from uqlab.data.class_regions import DEFAULT_FOUR_REGION_PRESET
from uqlab.evaluation.reporting.four_region_reporting import (
    four_region_signals_dataframe,
    list_four_region_signal_columns,
    plot_four_region_metrics_by_group,
)
from uqlab.runtime_paths import repository_root

if "RESULTS_BASE" not in globals():
    RESULTS_BASE = repository_root() / "results/four_region_benchmark"
if "RUNS" not in globals():
    ROOT = repository_root()
    RUNS = [
        {
            "name": "fashion_mlp",
            "config_path": ROOT / "configs/experiment/four_region_fashion_mlp.yaml",
            "class_regions": DEFAULT_FOUR_REGION_PRESET,
        },
        {
            "name": "cifar_resnet",
            "config_path": ROOT / "configs/experiment/four_region_cifar_resnet.yaml",
            "class_regions": DEFAULT_FOUR_REGION_PRESET,
        },
    ]
if "ATTRIBUTION_METRICS" not in globals():
    ATTRIBUTION_METRICS = [
        "inverse_coherence_dualxda",
        "inverse_coherence_ek_fak",
        "expected_entropy",
        "mutual_info",
        "inverse_logit_magnitude",
    ]

# Use in-memory results from cell 2, or rediscover finished runs on disk.
if "completed_dirs" not in globals() or not completed_dirs:
    completed_dirs = [
        RESULTS_BASE / spec["name"]
        for spec in RUNS
        if (RESULTS_BASE / spec["name"] / "per_sample_signals.csv").is_file()
    ]

if not completed_dirs:
    raise FileNotFoundError(
        "No finished runs found. Run cell 2 first, or check RESULTS_BASE / RUNS names."
    )

for run_dir in completed_dirs:
    df = four_region_signals_dataframe(run_dir)
    metrics = [m for m in ATTRIBUTION_METRICS if m in df.columns]
    if not metrics:
        metrics = list_four_region_signal_columns(df)
    out_dir = run_dir / "analysis" / "four_region_metrics"
    paths = plot_four_region_metrics_by_group(
        df,
        metrics,
        out_dir,
        title_prefix=run_dir.name,
    )
    print(run_dir.name, "→", len(paths), "plots under", out_dir)

ModuleNotFoundError: No module named 'uqlab'